# AusLAMP Victorian MT Data: Debug Problematic Sites

**Goal**: Diagnose and fix loading issues for sites that failed during batch processing.

**Problem sites identified**:
- VIC046: Only 20,400 samples (0.0 days)
- VIC049: Length mismatch error
- VIC054: Only 1,800 samples
- VIC059: Only 13,200 samples
- VIC060: Only 19,200 samples
- VIC061: Only 12,600 samples
- VIC063: Length mismatch error
- VIC068: Length mismatch error
- VIC074: Only 5,400 samples
- VIC076: Length mismatch error
- VIC078b: Length mismatch error
- VIC100: Only 600 samples

**Data**: EDL MiniSEED files from E:\MT_Timeseries_DATA\MT_AusLAMP_GA\EDL_raw

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import read
from IPython.display import Markdown, display

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

def caption(text):
    """Display a figure caption as wrapped, italic markdown."""
    display(Markdown(f"*{text}*"))

## Helper Functions for Detailed Diagnostics

In [2]:
def diagnose_site(station_dir):
    """
    Perform detailed diagnostics on a single site.
    
    Returns:
    - File inventory with sample counts
    - Channel consistency check
    - Time coverage analysis
    - File size analysis
    """
    station_id = station_dir.name
    print(f"\n{'='*80}")
    print(f"DIAGNOSING: {station_id}")
    print(f"{'='*80}\n")
    
    # Scan all day folders
    day_folders = sorted([d for d in station_dir.glob('*') if d.is_dir()])
    
    if not day_folders:
        print(f"ERROR: No day folders found in {station_dir}")
        return None
    
    print(f"Found {len(day_folders)} day folders\n")
    
    # Analyze files by channel
    channels = ['BX', 'BY', 'BZ', 'EX', 'EY', 'TP']
    file_info = {ch: [] for ch in channels}
    
    for day_folder in day_folders:
        day_num = day_folder.name
        
        for ch in channels:
            # Find all files for this channel
            files = sorted(list(day_folder.glob(f'*.{ch}')))
            
            for f in files:
                try:
                    # Read MiniSEED file
                    st = read(str(f), format='MSEED')
                    
                    if len(st) > 0:
                        tr = st[0]
                        file_info[ch].append({
                            'day': day_num,
                            'filename': f.name,
                            'filepath': str(f),
                            'n_samples': tr.stats.npts,
                            'sample_rate': tr.stats.sampling_rate,
                            'start_time': tr.stats.starttime,
                            'end_time': tr.stats.endtime,
                            'file_size': f.stat().st_size,
                            'duration_sec': (tr.stats.endtime - tr.stats.starttime)
                        })
                except Exception as e:
                    file_info[ch].append({
                        'day': day_num,
                        'filename': f.name,
                        'filepath': str(f),
                        'n_samples': None,
                        'sample_rate': None,
                        'start_time': None,
                        'end_time': None,
                        'file_size': f.stat().st_size,
                        'duration_sec': None,
                        'error': str(e)
                    })
    
    # Convert to DataFrames for analysis
    dfs = {}
    for ch in channels:
        if file_info[ch]:
            dfs[ch] = pd.DataFrame(file_info[ch])
    
    # Print summary
    print("\nFILE COUNT BY CHANNEL:")
    print("-" * 40)
    for ch in channels:
        n_files = len(file_info[ch])
        if n_files > 0:
            total_samples = sum([f['n_samples'] for f in file_info[ch] if f['n_samples'] is not None])
            n_errors = sum([1 for f in file_info[ch] if 'error' in f])
            print(f"{ch:3s}: {n_files:4d} files, {total_samples:12,} samples, {n_errors} errors")
        else:
            print(f"{ch:3s}: NO FILES FOUND")
    
    # Check for channel consistency
    print("\n\nCHANNEL CONSISTENCY CHECK:")
    print("-" * 40)
    file_counts = {ch: len(file_info[ch]) for ch in channels}
    max_count = max(file_counts.values())
    min_count = min(file_counts.values())
    
    if max_count == min_count:
        print(f"✓ All channels have same file count: {max_count}")
    else:
        print(f"✗ MISMATCH: File counts range from {min_count} to {max_count}")
        for ch in channels:
            if file_counts[ch] != max_count:
                print(f"  - {ch}: {file_counts[ch]} files (missing {max_count - file_counts[ch]})")
    
    # Check sample counts per file
    print("\n\nSAMPLE COUNT ANALYSIS:")
    print("-" * 40)
    for ch in channels:
        if ch in dfs and 'n_samples' in dfs[ch].columns:
            df = dfs[ch]
            valid = df[df['n_samples'].notna()]
            if len(valid) > 0:
                print(f"\n{ch}:")
                print(f"  Mean samples/file: {valid['n_samples'].mean():.0f}")
                print(f"  Min samples/file:  {valid['n_samples'].min():.0f}")
                print(f"  Max samples/file:  {valid['n_samples'].max():.0f}")
                print(f"  Std samples/file:  {valid['n_samples'].std():.0f}")
                
                # Flag unusual files
                mean_samples = valid['n_samples'].mean()
                std_samples = valid['n_samples'].std()
                outliers = valid[(valid['n_samples'] < mean_samples - 2*std_samples) | 
                                (valid['n_samples'] > mean_samples + 2*std_samples)]
                
                if len(outliers) > 0:
                    print(f"  ⚠ {len(outliers)} files with unusual sample counts:")
                    for idx, row in outliers.iterrows():
                        print(f"    - {row['filename']}: {row['n_samples']:.0f} samples")
    
    # File size analysis
    print("\n\nFILE SIZE ANALYSIS:")
    print("-" * 40)
    for ch in channels:
        if ch in dfs:
            df = dfs[ch]
            if len(df) > 0:
                print(f"\n{ch}:")
                print(f"  Mean size: {df['file_size'].mean()/1024:.1f} KB")
                print(f"  Min size:  {df['file_size'].min()/1024:.1f} KB")
                print(f"  Max size:  {df['file_size'].max()/1024:.1f} KB")
                
                # Expected size for 3600 samples @ 10 Hz = 49,152 bytes
                expected_size = 49152
                small_files = df[df['file_size'] < expected_size * 0.5]
                if len(small_files) > 0:
                    print(f"  ⚠ {len(small_files)} files smaller than expected:")
                    for idx, row in small_files.head(10).iterrows():
                        print(f"    - {row['filename']}: {row['file_size']/1024:.1f} KB")
    
    return dfs

## Diagnose Each Problem Site

In [3]:
# Base directory
base_dir = Path(r'E:\MT_Timeseries_DATA\MT_AusLAMP_GA\EDL_raw')

# List of problem sites
problem_sites = [
    'VIC046', 'VIC049', 'VIC054', 'VIC059', 'VIC060', 'VIC061',
    'VIC063', 'VIC068', 'VIC074', 'VIC076', 'VIC078b', 'VIC100'
]

# Store diagnostics results
diagnostics = {}

for site_id in problem_sites:
    station_dir = base_dir / site_id
    if station_dir.exists():
        diagnostics[site_id] = diagnose_site(station_dir)
    else:
        print(f"\nWARNING: {site_id} directory not found")


DIAGNOSING: VIC046

Found 22 day folders


FILE COUNT BY CHANNEL:
----------------------------------------
BX :  480 files,   17,280,000 samples, 0 errors
BY :  480 files,   17,280,000 samples, 0 errors
BZ :  480 files,   17,280,000 samples, 0 errors
EX :  480 files,   17,280,000 samples, 0 errors
EY :  480 files,   17,280,000 samples, 0 errors
TP :  480 files,   17,280,000 samples, 0 errors


CHANNEL CONSISTENCY CHECK:
----------------------------------------
✓ All channels have same file count: 480


SAMPLE COUNT ANALYSIS:
----------------------------------------

BX:
  Mean samples/file: 36000
  Min samples/file:  36000
  Max samples/file:  36000
  Std samples/file:  0

BY:
  Mean samples/file: 36000
  Min samples/file:  36000
  Max samples/file:  36000
  Std samples/file:  0

BZ:
  Mean samples/file: 36000
  Min samples/file:  36000
  Max samples/file:  36000
  Std samples/file:  0

EX:
  Mean samples/file: 36000
  Min samples/file:  36000
  Max samples/file:  36000
  Std samples/

## Summary of Issues

In [4]:
print("\n" + "="*80)
print("SUMMARY OF ISSUES")
print("="*80 + "\n")

for site_id in problem_sites:
    if site_id in diagnostics and diagnostics[site_id]:
        dfs = diagnostics[site_id]
        print(f"\n{site_id}:")
        
        # Check if all channels have data
        channels = ['BX', 'BY', 'BZ', 'EX', 'EY', 'TP']
        file_counts = {ch: len(dfs.get(ch, [])) for ch in channels}
        
        missing_channels = [ch for ch in channels if file_counts[ch] == 0]
        if missing_channels:
            print(f"  ✗ Missing channels: {', '.join(missing_channels)}")
        
        # Check for mismatched file counts
        counts = [c for c in file_counts.values() if c > 0]
        if len(set(counts)) > 1:
            print(f"  ✗ Channel file count mismatch: {file_counts}")
        
        # Total samples
        total_samples = {}
        for ch in channels:
            if ch in dfs and 'n_samples' in dfs[ch].columns:
                total_samples[ch] = dfs[ch]['n_samples'].sum()
        
        if total_samples:
            print(f"  Total samples: {total_samples}")
            
            # Check if sample counts match across channels
            sample_vals = list(total_samples.values())
            if len(set(sample_vals)) > 1:
                print(f"  ✗ Sample count mismatch across channels!")


SUMMARY OF ISSUES


VIC046:
  Total samples: {'BX': np.int64(17280000), 'BY': np.int64(17280000), 'BZ': np.int64(17280000), 'EX': np.int64(17280000), 'EY': np.int64(17280000), 'TP': np.int64(17280000)}

VIC049:
  Total samples: {'BX': np.int64(35959800), 'BY': np.int64(35959800), 'BZ': np.int64(35959800), 'EX': np.int64(35959800), 'EY': np.int64(35947800), 'TP': np.int64(35947800)}
  ✗ Sample count mismatch across channels!

VIC054:
  Total samples: {'BX': np.int64(28405800), 'BY': np.int64(28405800), 'BZ': np.int64(28405800), 'EX': np.int64(28405800), 'EY': np.int64(28405800), 'TP': np.int64(28405800)}

VIC059:
  Total samples: {'BX': np.int64(14289600), 'BY': np.int64(14289600), 'BZ': np.int64(14289600), 'EX': np.int64(14289600), 'EY': np.int64(14289600), 'TP': np.int64(14289600)}

VIC060:
  Total samples: {'BX': np.int64(12223200), 'BY': np.int64(12223200), 'BZ': np.int64(12223200), 'EX': np.int64(12223200), 'EY': np.int64(12223200), 'TP': np.int64(12223200)}

VIC061:
  Total sampl

## Detailed Investigation: Individual Sites

Look at specific problem sites in detail

In [ ]:
# Example: Investigate VIC049 in detail
site_id = 'VIC076'
if site_id in diagnostics and diagnostics[site_id]:
    print(f"Detailed investigation of {site_id}:\n")
    
    dfs = diagnostics[site_id]
    
    # Show file list for each channel
    for ch in ['BX', 'BY', 'BZ', 'EX', 'EY', 'TP']:
        if ch in dfs:
            print(f"\n{ch} files:")
            display(dfs[ch][['day', 'filename', 'n_samples', 'sample_rate', 'file_size']])

Detailed investigation of VIC049:


BX files:


,day,filename,n_samples,sample_rate,file_size
0,061,VIC49140302080000.BX,36000,10.0,53248
1,061,VIC49140302090000.BX,36000,10.0,53248
2,061,VIC49140302100000.BX,36000,10.0,53248
3,061,VIC49140302110000.BX,36000,10.0,53248
4,061,VIC49140302120000.BX,36000,10.0,53248
...,...,...,...,...,...
994,102,VIC49140412180000.BX,36000,10.0,49152
995,102,VIC49140412190000.BX,36000,10.0,53248
996,102,VIC49140412200000.BX,36000,10.0,53248
997,102,VIC49140412210000.BX,36000,10.0,53248



BY files:


,day,filename,n_samples,sample_rate,file_size
0,061,VIC49140302080000.BY,36000,10.0,53248
1,061,VIC49140302090000.BY,36000,10.0,53248
2,061,VIC49140302100000.BY,36000,10.0,53248
3,061,VIC49140302110000.BY,36000,10.0,53248
4,061,VIC49140302120000.BY,36000,10.0,53248
...,...,...,...,...,...
994,102,VIC49140412180000.BY,36000,10.0,49152
995,102,VIC49140412190000.BY,36000,10.0,53248
996,102,VIC49140412200000.BY,36000,10.0,53248
997,102,VIC49140412210000.BY,36000,10.0,53248



BZ files:


...

,day,filename,n_samples,sample_rate,file_size
0,061,VIC49140302080000.EX,36000,10.0,53248
1,061,VIC49140302090000.EX,36000,10.0,53248
2,061,VIC49140302100000.EX,36000,10.0,53248
3,061,VIC49140302110000.EX,36000,10.0,53248
4,061,VIC49140302120000.EX,36000,10.0,53248
...,...,...,...,...,...
994,102,VIC49140412180000.EX,36000,10.0,49152
995,102,VIC49140412190000.EX,36000,10.0,53248
996,102,VIC49140412200000.EX,36000,10.0,53248
997,102,VIC49140412210000.EX,36000,10.0,53248



EY files:


,day,filename,n_samples,sample_rate,file_size
0,061,VIC49140302080000.EY,36000,10.0,90112
1,061,VIC49140302090000.EY,36000,10.0,90112
2,061,VIC49140302100000.EY,36000,10.0,90112
3,061,VIC49140302110000.EY,36000,10.0,90112
4,061,VIC49140302120000.EY,36000,10.0,77824
...,...,...,...,...,...
994,102,VIC49140412180000.EY,36000,10.0,49152
995,102,VIC49140412190000.EY,36000,10.0,53248
996,102,VIC49140412200000.EY,36000,10.0,53248
997,102,VIC49140412210000.EY,36000,10.0,53248



TP files:


,day,filename,n_samples,sample_rate,file_size
0,061,VIC49140302080000.TP,36000,10.0,81920
1,061,VIC49140302090000.TP,36000,10.0,81920
2,061,VIC49140302100000.TP,36000,10.0,81920
3,061,VIC49140302110000.TP,36000,10.0,61440
4,061,VIC49140302120000.TP,36000,10.0,53248
...,...,...,...,...,...
994,102,VIC49140412180000.TP,36000,10.0,49152
995,102,VIC49140412190000.TP,36000,10.0,53248
996,102,VIC49140412200000.TP,36000,10.0,53248
997,102,VIC49140412210000.TP,36000,10.0,53248


## Proposed Solutions

Based on diagnostics, implement fixes for each type of issue:

1. **Missing channels**: Skip sites with incomplete channel sets
2. **Sample count mismatches**: Truncate to minimum length across channels
3. **Corrupted files**: Skip corrupted files, flag in metadata
4. **Very short recordings**: Flag as incomplete, may skip export

In [6]:
# This cell can be used to test fixes once issues are identified